<a href="https://colab.research.google.com/github/purveshvasantwagh-alt/digital-pathology-cell-analysis/blob/main/Cell_Segmentation_and_Classification_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision opencv-python-headless matplotlib scikit-learn grad-cam kagglehub

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import models, transforms
import kagglehub

print("Setup completed. GPU active:", torch.cuda.is_available())

In [ ]:
# Downloads Blood Cell Images dataset automatically
dataset_path = kagglehub.dataset_download("paultimothymooney/blood-cells")
print("Dataset location:", dataset_path)

In [ ]:
def process_cell_segmentation(image_path):
    # Read image and convert color spaces
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Noise reduction and adaptive thresholding
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological operations to clean background noise
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

    # Distance transform & Watershed segmentation for overlapping cell boundaries
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist_transform, 0.7 * dist_transform.max(), 255, 0)

    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    markers = cv2.watershed(img_rgb, markers)
    cell_count = len(np.unique(markers)) - 2

    return img_rgb, markers, cell_count

In [ ]:
# Initialize device and ResNet18 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze base feature extraction layers
for param in model.parameters():
    param.requires_grad = False

# Adapt classification head for 4 cell classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print("Transfer learning classifier built on GPU.")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

def generate_explainability_heatmap(model, input_tensor, original_img_rgb):
    target_layers = [model.layer4[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)

    targets = [ClassifierOutputTarget(0)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

    norm_img = np.float32(original_img_rgb) / 255.0
    visualization = show_cam_on_image(norm_img, grayscale_cam, use_rgb=True)
    return visualization